# 02 — Teste de inferência

Confere que o caminho completo funciona de ponta a ponta, numa amostra pequena:

**ler o dataset → montar o prompt congelado → gerar com o modelo → extrair a
letra → medir acurácia → gravar o JSONL.**

Não é um experimento. É o teste de fumaça que precisa passar antes de rodar o
baseline nos cinco datasets. Se algo aqui quebrar, quebra igual em escala, só
depois de horas de GPU.

Pré-requisitos:

```bash
python scripts/setup_datasets.py
python scripts/setup_models.py --models phi4-mini
# execute 01_formatacao_e_selecao.ipynb
```

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))            # pacote rmcq
sys.path.insert(0, str(ROOT / "scripts"))  # shims de compatibilidade

from rmcq.common import (
    build_answer_prompt,
    build_reflection_prompt,
    extract_final_answer,
    read_jsonl,
    write_jsonl,
)
from config import DATASETS, MODELS, RESULTS_DIR, SEED, SPLITS_DIR, STUDENT_GEN

pd.set_option("display.max_colwidth", 100)

# ---- parâmetros deste teste ----
DATASET = "arc"          # qualquer chave de config.DATASETS
SPLIT = "train"          # train (já selecionado por Cochran) ou test
MODEL = "phi4-mini"      # o menor dos quatro; troque para escalar o teste
N_QUESTIONS = 10
MAX_NEW_TOKENS = 640     # bem abaixo dos 4096 do experimento, só para ir rápido

print(f"dataset : {DATASET}/{SPLIT}  ({DATASETS[DATASET].problem_type})")
print(f"modelo  : {MODEL}  ->  {MODELS[MODEL].repo_id}  ({MODELS[MODEL].params})")
print(f"itens   : {N_QUESTIONS}")

## 1. Leitura do dataset

Todo dataset é lido do mesmo jeito, do mesmo lugar, com os mesmos campos. Era
esse o ponto do notebook 01.

In [ ]:
path = SPLITS_DIR / DATASET / f"{SPLIT}.jsonl"
if not path.exists():
    raise FileNotFoundError(f"{path} não existe. Execute 01_formatacao_e_selecao.ipynb primeiro.")

items = read_jsonl(path)
sample = items[:N_QUESTIONS]

print(f"{len(items):,} itens em {path.relative_to(ROOT)}")
pd.DataFrame([
    {
        "uid": i["uid"],
        "pergunta": i["question"][:70] + ("..." if len(i["question"]) > 70 else ""),
        "n_opções": i["num_choices"],
        "gabarito": i["answerKey"],
        "tem contexto": i["context"] is not None,
    }
    for i in sample
])

## 2. O prompt, exatamente como o modelo vai receber

Vale olhar antes de gastar GPU. É o `ANSWER_PROMPT` de `scripts/common.py`, sem
nenhuma adaptação por modelo ou por dataset.

In [ ]:
print(build_answer_prompt(sample[0]))
print("\n" + "=" * 78)
print(f"gabarito: {sample[0]['answerKey']}")

### O extrator, antes do modelo

Se o extrator estiver errado, a acurácia do experimento está errada e nada no
resto do pipeline avisa. Os casos abaixo cobrem o formato exigido e as três
degradações que aparecem na prática: markdown em volta do rótulo, resposta em
prosa, e bloco `<think>` de modelos de raciocínio (Qwen3) contendo uma letra
diferente da conclusão.

In [ ]:
extractor_cases = [
    ("formato exigido",        "Let me work through this.\n\nFINAL ANSWER: B",            "B"),
    ("markdown em volta",      "Reasoning here.\n\n**FINAL ANSWER:** (C)",                "C"),
    ("prosa",                  "After comparing them, the correct answer is D.",          "D"),
    ("letra solta",            "The reasoning points one way.\n\nA\n",                    "A"),
    ("<think> com outra letra", "<think>Maybe A works</think>\nActually no.\nFINAL ANSWER: C", "C"),
    ("repetido",               "FINAL ANSWER: A\nOn reflection, FINAL ANSWER: B",         "B"),
    ("abstenção",              "There is not enough information to decide.",              None),
]

rows = []
for name, text, expected in extractor_cases:
    ext = extract_final_answer(text, list("ABCD"))
    rows.append({
        "caso": name,
        "extraído": ext.letter,
        "esperado": expected,
        "método": ext.method,
        "seguiu formato": ext.followed_format,
        "ok": ext.letter == expected,
    })

extractor_df = pd.DataFrame(rows)
assert extractor_df["ok"].all(), "o extrator falhou em algum caso conhecido"
print("extrator: todos os casos passaram")
extractor_df

## 3. Carregamento do modelo

`ModelRunner` resolve dtype, `device_map`, `trust_remote_code` e chat template a
partir do `ModelSpec`. Um modelo de 8B em bfloat16 ocupa cerca de 16 GB de VRAM.

A primeira execução baixa os pesos se `setup_models.py` ainda não rodou.

In [ ]:
import torch

print(f"torch {torch.__version__}   cuda disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  cuda:{i}  {p.name}  {p.total_memory / 1024**3:.1f} GiB")
else:
    print("  AVISO: sem GPU, um modelo de 8B na CPU leva minutos por questão")

In [ ]:
from inference import ModelRunner, accuracy   # shim sobre rmcq.backends

runner = ModelRunner(MODEL)
print(f"carregado em: {runner.model.device}")
print(f"parâmetros  : {sum(p.numel() for p in runner.model.parameters()) / 1e9:.2f}B")

## 4. Uma questão, para inspecionar a saída bruta

Antes do laço, um item só, com a resposta completa impressa. É aqui que se vê se
o modelo raciocina passo a passo, se termina na linha exigida, e se há qualquer
coisa depois dela.

In [ ]:
record = runner.answer(sample[0], stage="smoke", condition="no_reflection",
                       max_new_tokens=MAX_NEW_TOKENS)

print(record.raw_output)
print("=" * 78)
print(f"extraído {record.predicted}  |  gabarito {record.gold}  |  "
      f"{'CERTO' if record.is_correct else 'ERRADO'}")
print(f"método '{record.extraction_method}'  seguiu formato: {record.followed_format}")
print(f"{record.prompt_tokens} tokens de entrada, {record.completion_tokens} de saída, "
      f"{record.latency_s}s")

## 5. O laço sobre a amostra

Sequencial e sem batching de propósito: o objetivo é validar corretude, não
throughput. O baseline de verdade vai precisar de batching ou vLLM.

In [ ]:
from tqdm.auto import tqdm

records = []
for item in tqdm(sample, desc=f"{MODEL} em {DATASET}/{SPLIT}"):
    records.append(
        runner.answer(item, stage="smoke", condition="no_reflection",
                      max_new_tokens=MAX_NEW_TOKENS)
    )

results_df = pd.DataFrame([
    {
        "uid": r.uid,
        "pred": r.predicted,
        "gold": r.gold,
        "certo": r.is_correct,
        "método": r.extraction_method,
        "tokens_saída": r.completion_tokens,
        "latência_s": r.latency_s,
    }
    for r in records
])
results_df

## 6. Métricas

Três números que merecem leitura separada:

- **`accuracy_strict`** conta abstenção como erro. É o número que vai para o paper.
- **`accuracy_answered`** ignora abstenções. A diferença entre os dois isola
  incapacidade de resolver de incapacidade de seguir o formato.
- **`format_adherence`** é a fração que terminou exatamente em `FINAL ANSWER: X`.
  Se cair muito para um modelo, isso é resultado a reportar, não bug a corrigir
  no extrator.

In [ ]:
metrics = accuracy(records)
for k, v in metrics.items():
    print(f"  {k:<26} {v}")

chance = sum(1 / i["num_choices"] for i in sample) / len(sample)
print(f"\n  {'acurácia do chute':<26} {chance:.4f}")
print(f"  {'acima do chute':<26} {metrics['accuracy_strict'] - chance:+.4f}")

### Erros e abstenções

Com 10 itens não há conclusão estatística nenhuma. O que se olha aqui é
qualitativo: o modelo errou porque raciocinou mal, ou porque a resposta certa
estava no texto e o extrator não pegou?

In [ ]:
problems = [r for r in records if not r.is_correct]
print(f"{len(problems)} de {len(records)} para inspecionar\n")

for r in problems[:3]:
    print("=" * 78)
    print(f"{r.uid}   pred={r.predicted}  gold={r.gold}  método='{r.extraction_method}'")
    print("-" * 78)
    print(r.raw_output.strip()[-700:])
    print()

## 7. Reflexão do professor sobre uma resposta errada

Um passo à frente, só para confirmar que o prompt de reflexão renderiza e que o
professor obedece a restrição central: **não revelar a resposta correta.** Se ele
revelar, a etapa de avaliação vaza o gabarito e o experimento inteiro perde
sentido.

Aqui aluno e professor são o mesmo modelo, o que é a condição de
autorreflexão. Na grade completa serão todos os pares viáveis.

In [ ]:
target = problems[0] if problems else records[0]
item = next(i for i in sample if i["uid"] == target.uid)

reflection = runner.reflect(
    item,
    previous_answer=target.raw_output,
    was_correct=bool(target.is_correct),
    depth="simple",
    perspective="teacher",
    max_new_tokens=400,
)

print(reflection.text.strip())
print("\n" + "=" * 78)
print(f"{len(reflection.text.split())} palavras, {reflection.completion_tokens} tokens, "
      f"{reflection.latency_s}s")

# Checagem de vazamento: a reflexão não deve nomear a alternativa correta.
gold_text = next(c["text"] for c in item["choices"] if c["label"] == item["answerKey"])
leaked_letter = extract_final_answer(reflection.text, [item["answerKey"]]).letter is not None
leaked_text = gold_text.lower() in reflection.text.lower()
print(f"vazou a letra do gabarito: {leaked_letter}")
print(f"vazou o texto do gabarito: {leaked_text}")

## 8. Gravação no formato único

Um JSONL, o mesmo `Record` que baseline, treino e avaliação vão usar. Colunas
que ainda não se aplicam ficam nulas em vez de ausentes, para que o arquivo
carregue direto num DataFrame sem alinhamento de schema.

In [ ]:
out_path = RESULTS_DIR / "smoke" / f"{MODEL}_{DATASET}_{SPLIT}.jsonl"
n = write_jsonl(out_path, records)
print(f"{n} linhas -> {out_path.relative_to(ROOT)}")

reloaded = pd.DataFrame(read_jsonl(out_path))
print(f"recarregado: {reloaded.shape[0]} linhas x {reloaded.shape[1]} colunas")
print(f"\ncolunas: {list(reloaded.columns)}")
reloaded[["uid", "dataset", "student_model", "predicted", "gold", "is_correct",
          "extraction_method", "completion_tokens", "latency_s"]].head()

In [ ]:
runner.unload()
if torch.cuda.is_available():
    print(f"VRAM alocada após liberar: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")

## O que este notebook não faz

Deliberadamente fora de escopo aqui, e necessário antes do baseline de verdade:

- **Batching.** Uma questão por vez desperdiça a GPU. `~1.800` questões de treino
  mais `~4.800` de teste, vezes quatro alunos, precisa de batching ou vLLM.
- **Retomada.** O baseline vai levar horas; precisa gravar incrementalmente e
  saber pular o que já fez, indexado por `uid`.
- **Troca de modelo em sequência.** `runner.unload()` existe para isso, mas o
  laço externo sobre os quatro modelos ainda não está escrito.
- **`max_new_tokens = 4096`.** Aqui usamos 640 para ir rápido. O experimento usa
  o valor do Caderno.

Rode este notebook uma vez com `MODEL = "phi4-mini"` e depois com cada um dos
outros três. Os pontos de atenção conhecidos: o Qwen3 tem modo de pensamento
híbrido (`QWEN_ENABLE_THINKING` em `config.py`) e o Llama 3 exige `HF_TOKEN`
com a licença aceita.